In [4]:
# ============================
# 1) IMPORTS
# ============================
import pandas as pd
import numpy as np
import re
import networkx as nx
from collections import Counter
from heapq import heappush, heappop


# ============================
# 2) LOAD DATASET
# ============================
df = pd.read_csv("last_clean_dataset1.csv")


# ============================
# 3) ARABIC EXTRACTION + NORMALIZATION
# ============================
arabic_pattern = re.compile(r'[\u0600-\u06FF]+')


def extract_arabic(text):
    if pd.isna(text):
        return ""
    return " ".join(arabic_pattern.findall(str(text)))


def normalize_arabic(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("[ًٌٍَُِّْ]", "", text)
    text = re.sub("ـ", "", text)
    text = re.sub("[^ء-ي\s]", " ", text)
    text = re.sub("\s+", " ", text).strip()
    return text


df["arabic_only"] = df["post_text"].apply(extract_arabic)
df["clean_text"] = df["arabic_only"].apply(normalize_arabic)

df = df[df["clean_text"].str.strip() != ""]
df = df.drop_duplicates(subset=["clean_text"])
df.reset_index(drop=True, inplace=True)


# ============================
# 4) DATETIME PROCESSING
# ============================
df["post_datetime"] = pd.to_datetime(df["post_datetime"], errors="coerce")
df["post_date"] = df["post_datetime"].dt.date
df["post_time"] = df["post_datetime"].dt.time
df["year"] = df["post_datetime"].dt.year
df["month"] = df["post_datetime"].dt.month
df["month_name"] = df["post_datetime"].dt.month_name()


# ============================
# 5) TOPIC RULES
# ============================
topic_rules = {
    "family": ["امي", "أمي", "ابوي", "ابويا", "اختي", "اخوي", "زوجي", "زوجتي", "عائلتي", "عائلة"],
    "education": ["جامعة", "مدرسة", "اختبار", "امتحان", "دراسة", "محاضرة", "واجب"],
    "work": ["وظيفة", "دوام", "مدير", "ترقية", "راتب", "شغل", "عمل"],
    "health": ["كورونا", "لقاح", "صحه", "صحة", "مستشفى"]
}


# ============================
# 6) TOPIC INFERENCE + FORWARD CHAINING
# ============================
def infer_topics(text):
    text = str(text)
    topics = []
    for topic, keywords in topic_rules.items():
        for kw in keywords:
            if kw in text:
                topics.append(topic)
                break
    if not topics:
        topics.append("unknown")
    return topics


def forward_chain_topics(text):
    return infer_topics(text)


df["inferred_topics"] = df["clean_text"].apply(infer_topics)
df["inferred_topics_fc"] = df["clean_text"].apply(forward_chain_topics)


# ============================
# 7) GRAPH CONSTRUCTION
# ============================
def text_to_words(text):
    return set(str(text).split())


def build_graph(df, min_shared_words=3):
    G = nx.Graph()
    for i in range(len(df)):
        G.add_node(i)

    texts = df["clean_text"].tolist()
    words_list = [text_to_words(t) for t in texts]

    for i in range(len(words_list)):
        for j in range(i+1, len(words_list)):
            shared = words_list[i].intersection(words_list[j])
            if len(shared) >= min_shared_words:
                G.add_edge(i, j, weight=1)
    return G


G = build_graph(df, min_shared_words=3)


# ============================
# 8) BFS SEARCH
# ============================
def bfs_related_posts(start_index, max_depth=2):
    visited = set()
    queue = [(start_index, 0)]
    related = []

    while queue:
        node, depth = queue.pop(0)
        if node in visited or depth > max_depth:
            continue
        visited.add(node)
        if node != start_index:
            related.append(node)
        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                queue.append((neighbor, depth + 1))
    return related


# ============================
# 9) A* SEARCH (BEST MATCH)
# ============================
def heuristic_overlap(query_words, post_words):
    return -len(query_words.intersection(post_words))


def a_star_best_match(query):
    query_words = text_to_words(normalize_arabic(query))
    best_score = float("inf")
    best_index = None

    for i, text in enumerate(df["clean_text"]):
        post_words = text_to_words(text)
        h = heuristic_overlap(query_words, post_words)
        if h < best_score:
            best_score = h
            best_index = i
    return best_index, best_score


# ============================
# 10) KNOWLEDGE / STATISTICS HELPERS
# ============================
def total_posts():
    return len(df)


def posts_per_year():
    return df["year"].value_counts().sort_index()


def posts_per_month():
    return df["month_name"].value_counts()


def keyword_frequency(keyword):
    return df["clean_text"].str.contains(keyword).sum()


def topic_distribution():
    return df["inferred_topics_fc"].apply(lambda x: tuple(x)).value_counts()


# ============================
# 11) SMART AGENT
# ============================
def smart_agent(query):
    q = query.lower().strip()

    if "how many posts" in q or "total posts" in q:
        return pd.DataFrame({"metric": ["total_posts"], "value": [total_posts()]})

    if "posts per year" in q:
        s = posts_per_year()
        return s.reset_index().rename(columns={"index": "year", "year": "count"})

    if "posts per month" in q:
        s = posts_per_month()
        return s.reset_index().rename(columns={"index": "month", "month_name": "count"})

    if q.startswith("topic "):
        topic = q.replace("topic", "").strip()
        mask = df["inferred_topics_fc"].apply(lambda lst: topic in lst)
        return df[mask][["author_name", "post_datetime", "clean_text", "inferred_topics_fc"]]

    if q.startswith("related "):
        try:
            idx = int(q.split()[1])
            related_idx = bfs_related_posts(idx)
            return df.loc[related_idx, ["author_name", "post_datetime", "clean_text"]]
        except:
            return pd.DataFrame({"error": ["Invalid index for related query"]})

    if q.startswith("best match "):
        user_q = q.replace("best match", "").strip()
        idx, score = a_star_best_match(user_q)
        row = df.loc[idx, ["author_name", "post_datetime", "clean_text"]]
        return pd.DataFrame([{
            "best_index": idx,
            "score": score,
            "author_name": row["author_name"],
            "post_datetime": row["post_datetime"],
            "clean_text": row["clean_text"]
        }])

    if q.startswith("count "):
        kw = q.replace("count", "").strip()
        c = keyword_frequency(kw)
        return pd.DataFrame({"keyword": [kw], "count": [c]})

    if "topic distribution" in q:
        dist = topic_distribution()
        return dist.reset_index().rename(columns={"index": "topic", 0: "count"})

    idx, score = a_star_best_match(q)
    row = df.loc[idx, ["author_name", "post_datetime", "clean_text"]]
    return pd.DataFrame([{
        "best_index": idx,
        "score": score,
        "author_name": row["author_name"],
        "post_datetime": row["post_datetime"],
        "clean_text": row["clean_text"]
    }])


# ============================
# 12) EXAMPLE QUERIES
# ============================
print(smart_agent("topic family").head())
print(smart_agent("topic education").head())
print(smart_agent("topic work").head())
print(smart_agent("topic unknown").head())
print(smart_agent("how many posts"))
print(smart_agent("posts per year"))
print(smart_agent("topic distribution"))
print(smart_agent("best match جامعة"))
print(smart_agent("related 0"))

# ============================
# 13) INTERACTIVE DEMO (For the Jury)
# ============================
from IPython.display import display

print("--- 🤖 SAUDI SOCIAL AI AGENT ---")
user_input = input("Enter your query (e.g., 'topic health', 'best match جامعة', or 'count كورونا'): ")

# The Agent processes the input and acts
result = smart_agent(user_input)

print("\n--- 🧠 AGENT REASONING RESULT ---")
display(result)


<>:37: SyntaxWarning: invalid escape sequence '\s'
<>:38: SyntaxWarning: invalid escape sequence '\s'
<>:37: SyntaxWarning: invalid escape sequence '\s'
<>:38: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_26909/1111005354.py:37: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("[^ء-ي\s]", " ", text)
/tmp/ipykernel_26909/1111005354.py:38: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("\s+", " ", text).strip()


   author_name       post_datetime  \
5   Reda Helal 2011-03-21 00:37:00   
7   Reda Helal 2011-04-10 16:36:00   
26  Reda Helal 2012-03-27 22:56:00   
28  Reda Helal 2012-05-11 18:47:00   
34  Reda Helal 2012-06-17 12:18:00   

                                           clean_text inferred_topics_fc  
5   صحفيون واعلاميون ورؤساء تحرير الصحف ووكالة الا...           [family]  
7   انه كبير الحرامية وزعيم العصابة الذي يضحي برجا...           [family]  
26  يارب فرج هموم امة نبيك محمد فقلوبنا ما عادت تح...           [family]  
28  وزير الخارجية يزور ابوظبي الثلاثاء لمباحثات ثن...           [family]  
34  يا من صدقتم اعلام مبارك الفاسد يا من صدقتم فضا...           [family]  
    author_name       post_datetime  \
46   Reda Helal 2012-10-30 00:48:00   
134  Reda Helal                 NaT   
148  Reda Helal                 NaT   

                                            clean_text  \
46   انتقل الى رحمة الله تعالى منذ ساعات والد زوجتي...   
134  كبسولة زكاة الفطر فرضت زكاة الفطر طهرة للصا

,best_index,score,author_name,post_datetime,clean_text
0,0,0,Reda Helal,2010-06-19 18:50:00,لا يكفي ان تكون في النور لكي تري بل ينبغي ان ي...
